In [33]:
%pip install -q shap

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from pathlib import Path

DATASET_NAME = "ObesityDataSet_raw_and_data_sinthetic.csv"

candidate_paths = [
    # Notebook is executed inside the notebooks folder
    Path("../") / DATASET_NAME,

    # Notebook is executed from the repository root
    Path(DATASET_NAME),
]

local_path = next(
    (path for path in candidate_paths if path.exists()),
    None
)

if local_path is not None:
    df = pd.read_csv(local_path)
    print("Dataset loaded from:", local_path.resolve())
else:
    # Fallback for opening the notebook directly in Google Colab
    DATA_URL = (
        "https://raw.githubusercontent.com/"
        "hee289427-wq/obesity-xai-prediction/main/"
        "ObesityDataSet_raw_and_data_sinthetic.csv"
    )
    df = pd.read_csv(DATA_URL)
    print("Dataset loaded from GitHub.")

print("Dataset shape:", df.shape)
df.head()

# Encode categorical features
label_encoders = {}
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Split features and target
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

weighted_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

print("Accuracy:", accuracy)
print("Weighted F1-score:", f1)
print("Weighted AUC:", weighted_auc)


Dataset loaded from GitHub.
Dataset shape: (2111, 17)
Accuracy: 0.9574468085106383
Weighted F1-score: 0.9580244534544385
Weighted AUC: 0.997346278387296


In [35]:
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Weighted F1", "Weighted AUC"]
values = [accuracy, f1, weighted_auc]

plt.figure(figsize=(6,4))
plt.bar(metrics, values)
plt.ylim(0, 1)
plt.title("Random Forest Performance Metrics")
plt.ylabel("Score")
plt.show()



In [36]:
def predict_obesity(person_dict, rf, label_encoders):
    # Create DataFrame
    df_input = pd.DataFrame([person_dict])

    # Encode categorical columns
    for col in df_input.columns:
        if col in label_encoders:
            df_input[col] = label_encoders[col].transform(df_input[col])

    # Ensure the same feature order used during model training
    df_input = df_input[X.columns]

    # Predict
    pred_class = rf.predict(df_input)[0]
    pred_label = label_encoders[
        "NObeyesdad"
    ].inverse_transform([pred_class])[0]

    return pred_label

In [37]:
import numpy as np

# Randomly select a test sample from the test set
idx = 0
sample = X_test.iloc[idx]

# Decode into readable human characteristics
def decode_sample(sample, label_encoders):
    decoded = {}

    for col in sample.index:
        val = sample[col]
        if col in label_encoders:
            decoded[col] = label_encoders[col].inverse_transform([int(val)])[0]
        else:
            decoded[col] = float(val)

    return decoded

human_sample = decode_sample(sample, label_encoders)

# Obtain the actual obesity level of this person (the "Ground Truth")
true_label_code = y_test.iloc[idx]
true_label = label_encoders["NObeyesdad"].inverse_transform([true_label_code])[0]

# Use the model to predict this person
predicted_label = predict_obesity(human_sample, rf, label_encoders)

# Formatting the numerical values makes it easier to read.
def pretty(v):
    if isinstance(v, float):
        return round(v, 2)
    return v

pretty_sample = {k: pretty(v) for k, v in human_sample.items()}

# Input
print("Test sample (human-readable):")
for k, v in pretty_sample.items():
    print(f"{k:30s}: {v}")


print("\nTrue obesity level:", true_label)
print("Predicted obesity level:", predicted_label)



Test sample (human-readable):
Gender                        : Female
Age                           : 19.08
Height                        : 1.77
Weight                        : 49.6
family_history_with_overweight: no
FAVC                          : yes
FCVC                          : 3.0
NCP                           : 3.73
CAEC                          : Sometimes
SMOKE                         : no
CH2O                          : 1.19
SCC                           : no
FAF                           : 2.0
TUE                           : 1.0
CALC                          : Sometimes
MTRANS                        : Public_Transportation

True obesity level: Insufficient_Weight
Predicted obesity level: Insufficient_Weight


In [38]:
person = {
    "Gender": "Female",
    "Age": 25,
    "Height": 1.70,
    "Weight": 70,
    "family_history_with_overweight": "no",
    "FAVC": "no",
    "FCVC": 2,
    "NCP": 3,
    "CAEC": "Sometimes",
    "SMOKE": "no",
    "CH2O": 2,
    "SCC": "no",
    "FAF": 1,
    "TUE": 1,
    "CALC": "Sometimes",
    "MTRANS": "Walking"
}

manual_prediction = predict_obesity(
    person,
    rf,
    label_encoders
)

print("Manual profile prediction:", manual_prediction)

Manual profile prediction: Normal_Weight


In [39]:
class_names = label_encoders["NObeyesdad"].classes_
class_names


array(['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I',
       'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I',
       'Overweight_Level_II'], dtype=object)

In [40]:
print("Accuracy:", accuracy, type(accuracy))
print("Weighted F1:", f1, type(f1))
print("Weighted AUC:", weighted_auc, type(weighted_auc))

Accuracy: 0.9574468085106383 <class 'float'>
Weighted F1: 0.9580244534544385 <class 'float'>
Weighted AUC: 0.997346278387296 <class 'numpy.float64'>


In [41]:
from sklearn.metrics import roc_curve, auc as calculate_auc

plt.figure(figsize=(8, 6))

for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(
        y_test == i,
        y_prob[:, i]
    )

    class_auc = calculate_auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{class_names[i]} (AUC = {class_auc:.2f})"
    )

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Obesity Level Prediction")
plt.legend()
plt.tight_layout()
plt.show()

In [42]:
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(7,6))

for i in range(len(class_names)):
    precision, recall, _ = precision_recall_curve(y_test == i, y_prob[:, i])
    plt.plot(recall, precision, label=class_names[i])

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves for Obesity Levels")
plt.legend()
plt.show()



In [43]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1) 先算 confusion matrix
cm = confusion_matrix(y_test, y_pred)

# 2) 类别名字
class_names = label_encoders["NObeyesdad"].classes_

# 3) 画图
fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap="viridis")

ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

plt.title("Confusion Matrix for Obesity Level Prediction")
plt.tight_layout()
plt.show()



# Explainable AI Analysis

## SHAP Local Explanation
This section uses a SHAP waterfall plot to explain the feature
contributions behind an individual Random Forest prediction.

In [44]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Rebuild the explainer
explainer = shap.TreeExplainer(rf)

# 2. Use the same sample
X_sample = pd.DataFrame([sample], columns=X.columns)

# 3. Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

print("SHAP values shape:", np.asarray(shap_values).shape)
print("Expected value shape:", np.asarray(explainer.expected_value).shape)

# 4. Get the predicted class index
proba = rf.predict_proba(X_sample)[0]
class_idx = int(np.argmax(proba))
predicted_class_code = rf.classes_[class_idx]

predicted_class = label_encoders[
    "NObeyesdad"
].inverse_transform(
    [predicted_class_code]
)[0]

# 5. Build a single-class Explanation object
shap_explanation = shap.Explanation(
    values=shap_values[0, :, class_idx],
    base_values=explainer.expected_value[class_idx],
    data=X_sample.iloc[0].values,
    feature_names=X_sample.columns.tolist()
)

# 6. Plot the waterfall chart
shap.plots.waterfall(
    shap_explanation,
    max_display=15,
    show=False
)

plt.title(
    f"SHAP Waterfall Plot – Predicted Class: {predicted_class}",
    pad=20
)
plt.tight_layout()
plt.show()

SHAP values shape: (1, 16, 7)
Expected value shape: (7,)


## What-If Counterfactual Analysis
This section evaluates how changes in actionable variables affect
the predicted obesity level while the remaining features are fixed.

In [45]:
# Take one real test sample
sample = X_test.iloc[0].copy()

# Predict original
pred_class = rf.predict(pd.DataFrame([sample]))[0]
pred_label = label_encoders["NObeyesdad"].inverse_transform([pred_class])[0]

print("Original obesity prediction:", pred_label)


# What-if counterfactual changes

weight_change = 25
faf_change = 1
fcvc_change = 1
ch2o_change = 1

sample_cf = sample.copy()

# Change four actionable features
sample_cf["Weight"] = max(
    1,
    sample_cf["Weight"] + weight_change
)

sample_cf["FAF"] = np.clip(
    sample_cf["FAF"] + faf_change,
    0,
    3
)

sample_cf["FCVC"] = np.clip(
    sample_cf["FCVC"] + fcvc_change,
    1,
    3
)

sample_cf["CH2O"] = np.clip(
    sample_cf["CH2O"] + ch2o_change,
    1,
    3
)

# Predict counterfactual
new_class = rf.predict(pd.DataFrame([sample_cf]))[0]
new_label = label_encoders["NObeyesdad"].inverse_transform([new_class])[0]

print(f"After what-if changes (Weight {weight_change:+}, FAF {faf_change:+}, FCVC {fcvc_change:+}, CH2O {ch2o_change:+}):", new_label)

# Decode gender
gender_code = int(sample["Gender"])
gender_label = label_encoders["Gender"].inverse_transform([gender_code])[0]
print("Gender:", gender_label)


Original obesity prediction: Insufficient_Weight
After what-if changes (Weight +25, FAF +1, FCVC +1, CH2O +1): Normal_Weight
Gender: Female


In [46]:
import pandas as pd

cf_table = pd.DataFrame({
    "Scenario": ["Original", "Counterfactual (what-if)"],
    "Weight": [round(sample["Weight"], 2), round(sample_cf["Weight"], 2)],
    "FAF": [round(sample["FAF"], 2), round(sample_cf["FAF"], 2)],
    "FCVC": [round(sample["FCVC"], 2), round(sample_cf["FCVC"], 2)],
    "CH2O": [round(sample["CH2O"], 2), round(sample_cf["CH2O"], 2)],
    "Predicted Obesity Level": [pred_label, new_label]
})

cf_table



,Scenario,Weight,FAF,FCVC,CH2O,Predicted Obesity Level
0,Original,49.6,2.0,3.0,1.19,Insufficient_Weight
1,Counterfactual (what-if),74.6,3.0,3.0,2.19,Normal_Weight


## Interactive Web Application

The trained Random Forest model, SHAP explanation module, and
what-if counterfactual analysis were integrated into an interactive
Gradio web application.

The complete application is implemented in:

`../app.py`

To run the application locally from the project root directory:

```bash
pip install -r requirements.txt
python app.py
```

### Live Demo

https://obesity-xai-prediction.onrender.com

The web application is provided for educational machine-learning
demonstration and does not constitute medical diagnosis or health advice.